# Deep Learning for NLP

Name : Maan A. Alghamdi  
ID : 2240001433

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [1]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000
Out[1]: 
                                              review  label
0                           Wow... Loved this place.      1
1                                 Crust is not good.      0
2          Not tasty and the texture was just nasty.      0
3  Stopped by during the late May bank holiday of...      1
4  The selection on the menu was great and so wer...      1


In [2]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [3]:
import re

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

# Apply preprocessing outside the function, after its definition.
df["review"] = df["review"].apply(preprocess_text)
df.head()

Out[3]: 
                                              review  label
0                               wow loved this place      1
1                                  crust is not good      0
2           not tasty and the texture was just nasty      0
3  stopped by during the late may bank holiday of...      1
4  the selection on the menu was great and so wer...      1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [4]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [6]:
import torch
import torch.nn as nn

# Fix initialization and CPU execution for repeatable runs.
torch.manual_seed(42)
torch.set_num_threads(1)

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.tensor(y_train.to_numpy(copy=True), dtype=torch.float32).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [7]:
input_size = X_train_tensor.shape[1]
# Two hidden layers: 64 and 32 neurons.

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [8]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [9]:
epochs = 10

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()  # Clear gradients before each backward pass.

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6242
Epoch 10/10 - Loss: 0.3577


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.755


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [11]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.81      0.67      0.73       100
    Positive       0.72      0.84      0.77       100

    accuracy                           0.76       200
   macro avg       0.76      0.76      0.75       200
weighted avg       0.76      0.76      0.75       200

Confusion Matrix:
[[67 33]
 [16 84]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


### My approach
I tokenize the cleaned reviews, train skip-gram Word2Vec only on the training set, and average the known word vectors to obtain one 200-dimensional vector per review. I ignore unseen test words and use a zero vector when no words are known. This keeps test data out of embedding training.


In [12]:
import numpy as np
import zlib
from gensim.models import Word2Vec

# Whitespace tokenization is sufficient after the preprocessing above.
train_tokens = [review.split() for review in X_train]
test_tokens = [review.split() for review in X_test]

# A stable hash and a single worker make initialization reproducible.
def stable_hash(word):
    return zlib.crc32(word.encode("utf-8"))

# Learn the vocabulary and embeddings from training reviews only.
word2vec = Word2Vec(
    sentences=train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1,
    seed=42,
    workers=1,
    hashfxn=stable_hash,
    epochs=5,
)

def review_vector(tokens):
    known_vectors = [word2vec.wv[word] for word in tokens if word in word2vec.wv]
    if not known_vectors:
        return np.zeros(word2vec.vector_size, dtype=np.float32)
    return np.mean(known_vectors, axis=0).astype(np.float32)

X_train_w2v = np.stack([review_vector(tokens) for tokens in train_tokens])
X_test_w2v = np.stack([review_vector(tokens) for tokens in test_tokens])

X_train_w2v_tensor = torch.tensor(X_train_w2v, dtype=torch.float32)
X_test_w2v_tensor = torch.tensor(X_test_w2v, dtype=torch.float32)
y_train_w2v_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
y_test_w2v_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).reshape(-1, 1)

print("Word2Vec vocabulary size:", len(word2vec.wv))
print("Training Word2Vec tensor shape:", tuple(X_train_w2v_tensor.shape))
print("Testing Word2Vec tensor shape:", tuple(X_test_w2v_tensor.shape))
print("Training label tensor shape:", tuple(y_train_w2v_tensor.shape))
print("Testing label tensor shape:", tuple(y_test_w2v_tensor.shape))

Word2Vec vocabulary size: 1787
Training Word2Vec tensor shape: (800, 200)
Testing Word2Vec tensor shape: (200, 200)
Training label tensor shape: (800, 1)
Testing label tensor shape: (200, 1)


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [13]:
torch.manual_seed(42)

# Input: 200 features; hidden layers: 128, 64, 32; output: one logit.
w2v_model = nn.Sequential(
    nn.Linear(X_train_w2v_tensor.shape[1], 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
)
print(w2v_model)

w2v_loss_function = nn.BCEWithLogitsLoss()
w2v_optimizer = torch.optim.Adam(w2v_model.parameters(), lr=0.001)
w2v_epochs = 15
w2v_losses = []

for epoch in range(w2v_epochs):
    w2v_model.train()
    w2v_optimizer.zero_grad()
    logits = w2v_model(X_train_w2v_tensor)
    loss = w2v_loss_function(logits, y_train_w2v_tensor)
    loss.backward()
    w2v_optimizer.step()
    w2v_losses.append(loss.item())
    print(f"Epoch {epoch + 1}/{w2v_epochs} - Loss: {loss.item():.4f}")

w2v_model.eval()
with torch.no_grad():
    w2v_test_logits = w2v_model(X_test_w2v_tensor)
    w2v_probabilities = torch.sigmoid(w2v_test_logits)
    w2v_predictions = (w2v_probabilities >= 0.5).int().numpy().flatten()

w2v_accuracy = accuracy_score(y_test, w2v_predictions)
print(f"Word2Vec Test Accuracy: {w2v_accuracy:.3f} ({w2v_accuracy:.1%})")
print("Classification Report:")
print(classification_report(y_test, w2v_predictions,
                            labels=[0, 1], target_names=["Negative", "Positive"],
                            zero_division=0))
print("Confusion Matrix (rows = actual, columns = predicted; order = 0, 1):")
print(confusion_matrix(y_test, w2v_predictions, labels=[0, 1]))
print(f"TF-IDF baseline Test Accuracy: {accuracy:.3f} ({accuracy:.1%})")

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)
Epoch 1/15 - Loss: 0.6958
Epoch 2/15 - Loss: 0.6953
Epoch 3/15 - Loss: 0.6949
Epoch 4/15 - Loss: 0.6946
Epoch 5/15 - Loss: 0.6943
Epoch 6/15 - Loss: 0.6941
Epoch 7/15 - Loss: 0.6938
Epoch 8/15 - Loss: 0.6937
Epoch 9/15 - Loss: 0.6935
Epoch 10/15 - Loss: 0.6934
Epoch 11/15 - Loss: 0.6933
Epoch 12/15 - Loss: 0.6933
Epoch 13/15 - Loss: 0.6932
Epoch 14/15 - Loss: 0.6932
Epoch 15/15 - Loss: 0.6932
Word2Vec Test Accuracy: 0.490 (49.0%)
Classification Report:
              precision    recall  f1-score   support

    Negative       0.49      0.98      0.66       100
    Positive       0.00      0.00      0.00       100

    accuracy                           0.49       200
   macro avg       0.25      0.49  

## My results and conclusion

I obtained **75.5% test accuracy** with the TF-IDF baseline and **49.0%** with the Word2Vec network. The Word2Vec confusion matrix was [[98, 2], [100, 0]] (actual rows, predicted columns; negative then positive). It predicted almost every review as negative, and its final training loss of approximately 0.6932 indicates poor learning under this configuration.

I trained embeddings on only 800 short reviews and averaged their word vectors, which loses word order. These limitations may contribute to the weak result. I preserved the required three hidden layers, learning rate of 0.001, and 15 epochs. Any future tuning should use a separate validation set rather than the test set.